Plug your Huggingface token!

In [ ]:
huggingface_token = ""

In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U git+https://github.com/huggingface/transformers.git

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
!pip install evaluate rouge_score absl-py

In [ ]:

# 1. Adapter path
adapter_path = "/kaggle/input/notebooks/robinllee/dial2note-training/qwen_7"
base_model_name = "Qwen/Qwen3.5-0.8B"

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Model loading
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# 4. PEFT adapter integration
model = PeftModel.from_pretrained(base_model, adapter_path)

# 5. Inference mode
model.eval()

print("Model + adapter successful")


In [ ]:
from huggingface_hub import login
login(token=huggingface_token)

from datasets import load_from_disk, Dataset, load_dataset
ds = load_dataset("Ahmad0067/MedSynth")

initial_count = len(ds["train"])

# 1. None & missing value removal
ds = ds.filter(lambda x: 
    x["Dialogue"] is not None and 
    x[" Note"] is not None and 
    x["ICD10"] is not None
)

# 2. Too short dialogue or note (except placeholders < 50) 
min_length = 50 
ds = ds.filter(lambda x: 
    len(str(x["Dialogue"]).strip()) > min_length and 
    len(str(x[" Note"]).strip()) > min_length
)

# 3. de-duplication (based on dialogue)
df = ds["train"].to_pandas()
initial_count = len(df)
df = df.drop_duplicates(subset=["Dialogue"], keep="first")

print(f"before preprocessing: {initial_count} -> after preprocessing: {len(df)}개")

# get back to the form of Dataset
ds = Dataset.from_dict(df)

In [ ]:
instruction = (
    "Task: Convert the medical dialogue into a professional SOAP note.\n"
    "Guidelines:\n"
    "1. STRUCTURE:\n"
    " - Subjective: Patient-reported information (CC, HPI, ROS).\n"
    " - Objective: Observable and measurable findings from physical examination.\n"
    " - Assessment: Clinical diagnosis and medical reasoning.\n"
    " - Plan: Treatment recommendations, prescriptions, and follow-up care.\n"
    "2. CONSTRAINTS:\n"
    " - Use formal clinical terminology.\n"
    " - Use ONLY facts from the dialogue. DO NOT invent names, ages, or data.\n"
    " - Match gender pronouns and anatomical lateralization (Right/Left) strictly.\n"
    " - Start immediately with '**1. Subjective:**'."
)

In [ ]:
def get_predictions_fast(model, tokenizer, dialogue_list, batch_size=8):
    global instruction
    model.eval()
    all_predictions = []
    # batch feeding
    for i in range(0, len(dialogue_list), batch_size):
        batch_texts = dialogue_list[i : i + batch_size]
        
        # Prompt
        prompts = [f"{instruction}\n\nDialogue:\n{d}\n\nOutput:\n" for d in batch_texts]
        
        # Tokenization (padding = True)
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False, # 속도를 위해 False 권장
                repetition_penalty=1.2,
                pad_token_id=tokenizer.pad_token_id
            )
        
        # Decoding outputs
        decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        for full_text in decoded_outputs:
            pred = full_text.split("Output:")[-1].strip() if "Output:" in full_text else full_text
            all_predictions.append(pred)

        print(f"Batch ended. {(i+1)}/{len(dialogue_list)}")
            
    return all_predictions

In [ ]:
test_ds = ds.shuffle(seed=42)
total_count = len(ds)
# test_set 1000
test_ds = test_ds.select(range(total_count - 1000, total_count))
references = test_ds[" Note"]
dialogues = test_ds['Dialogue']

In [ ]:
predictions = get_predictions_fast(model, tokenizer, dialogues, batch_size=32)
print("Finished generation")

In [ ]:
import evaluate

# Loading the metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")

def calculate_all_metrics(predictions, references):
    # ROUGE-1, ROUGE-2, ROUGE-L
    rouge_results = rouge_metric.compute(
        predictions=predictions, 
        references=references, 
        use_stemmer=True
    )
    
    # BLEU
    bleu_results = bleu_metric.compute(
        predictions=predictions, 
        references=references
    )
    
    # METEOR
    meteor_results = meteor_metric.compute(
        predictions=predictions, 
        references=references
    )
    
    print("\n" + "="*40)
    print("📋 Final Performance Metrics")
    print("="*40)
    print(f"🔹 BLEU     : {bleu_results['bleu']:.4f}")
    print(f"🔹 ROUGE-1 : {rouge_results['rouge1']:.4f}")
    print(f"🔹 ROUGE-2 : {rouge_results['rouge2']:.4f}")
    print(f"🔹 ROUGE-L : {rouge_results['rougeL']:.4f}")
    print(f"🔹 METEOR  : {meteor_results['meteor']:.4f}")
    print("="*40)

# 실행
calculate_all_metrics(predictions, references)

# prediction and reference pair check
clean_predictions = [pred.replace('\\n', '\n') for pred in predictions]
print("Prediction: ")
print(clean_predictions[5])
print("####################################################")
print("####################################################")
print("####################################################")
print("Reference: ")
clean_references = [ref.replace('\\n', '\n') for ref in references]
print(clean_references[5])

import pandas as pd
df = pd.DataFrame({
    "predictions": clean_predictions,
    "references": clean_references
})

# 2. CSV로 저장
df.to_csv("inference.csv", index=False, encoding="utf-8-sig")
print("두 리스트가 각각의 컬럼으로 저장되었습니다!")